In [11]:
import numpy as np

In [16]:
b_sz = 8
x_sz = 5
y1_sz = 16
y2_sz = 2  # aka. y2_sz == y_true_sz

In [33]:
class Net:
    def __init__(self):
        self.X = np.random.randn(b_sz, x_sz)
        print(f"X: {self.X.dtype} {self.X.shape}")

        self.W1 = np.random.randn(x_sz, y1_sz)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")

        self.Z1 = np.zeros((b_sz, y1_sz))
        print(f"Z1: {self.Z1.dtype} {self.Z1.shape}")

        self.Y1 = np.zeros_like(self.Z1)
        print(f"Y1: {self.Y1.dtype} {self.Y1.shape}")

        self.W2 = np.random.randn(y1_sz, y2_sz)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")

        self.Z2 = np.zeros((b_sz, y2_sz))
        print(f"Z2: {self.Z2.dtype} {self.Z2.shape}")

        self.Y2 = np.zeros_like(self.Z2)
        print(f"Y2: {self.Y2.dtype} {self.Y2.shape}")

    @staticmethod
    def _f1(A):  # ReLU
        return A * (A > 0)

    @staticmethod
    def _df1(A):  # dReLU
        return (A > 0).astype(A.dtype)

    @staticmethod
    def _f2(A):  # linear / identity
        return A

    @staticmethod
    def _df2(A):  # d(linear) == 1
        return np.ones_like(A)

    def forward(self, X):
        self.X = X
        self.Z1 = np.dot(self.X, self.W1)
        self.Y1 = self._f1(self.Z1)
        self.Z2 = np.dot(self.Y1, self.W2)
        self.Y2 = self._f2(self.Z2)
        return self.Y2

    @staticmethod
    def _loss(Y_predicted, Y_true):  # 1/(2N) * Sum (y-y_true)^2, mean over batch
        return 0.5 * ((Y_predicted - Y_true) ** 2).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def backward(self, Y_true, lr=1e-3):
        self.E2 = self._dloss(self.Y2, Y_true) * self._df2(self.Z2)
        self.E1 = self.E2 @ self.W2.T * self._df1(self.Z1)

        self.dL_dW2 = self.Y1.T @ self.E2
        self.dL_dW1 = self.X.T @ self.E1

        self.W2 += -lr * self.dL_dW2
        self.W1 += -lr * self.dL_dW1


nn = Net()

X: float64 (8, 5)
W1: float64 (5, 16)
Z1: float64 (8, 16)
Y1: float64 (8, 16)
W2: float64 (16, 2)
Z2: float64 (8, 2)
Y2: float64 (8, 2)
